# Structure-Aware Semantic-Guided Thermal to Visible Image Translation
## Training Pipeline on LLVIP Dataset (Kaggle GPU Environment)

This notebook runs the complete 3-Stage training framework:
1. **Stage 1**: Pre-train Structure Extraction Network (Module 3A)
2. **Stage 2**: Train Feature Fusion (Module 4) + Appearance Generator (Module 5)
3. **Stage 3**: End-to-End Joint Fine-Tuning
4. **Evaluation**: PSNR, SSIM, and Visual Comparison Grid Generation

In [1]:
# Cell 1: Download your code from GitHub into Kaggle
!git clone https://github.com/Akhila-Bijja/Major-project.git
%cd Major-project


Cloning into 'Major-project'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 49 (delta 14), reused 38 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 26.63 KiB | 4.44 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/kaggle/working/Major-project


In [2]:
import os
import zipfile
import glob

# 1. Find the uploaded zip file in /kaggle/input/
zip_files = glob.glob("/kaggle/input/**/*.zip", recursive=True)

if not zip_files:
    print("❌ No .zip file found in /kaggle/input/. Please check your dataset attachment.")
else:
    zip_path = zip_files[0]
    print(f" Found dataset zip file: {zip_path}")
    
    # 2. Unzip into /kaggle/working/data/
    extract_dir = "/kaggle/working/data"
    os.makedirs(extract_dir, exist_ok=True)
    
    print("⏳ Unzipping dataset... (this may take 1-2 minutes)")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("✅ Unzipping Complete!")
    
    # 3. List extracted folders to verify
    print("\nExtracted contents:")
    for root, dirs, files in os.walk(extract_dir):
        level = root.replace(extract_dir, '').count(os.sep)
        if level <= 2:
            print(f"{'  ' * level}{os.path.basename(root)}/")


❌ No .zip file found in /kaggle/input/. Please check your dataset attachment.


In [3]:
# Clone project repository from GitHub
!git clone https://github.com/Akhila-Bijja/Major-project.git
%cd Major-project

# Automatically find the path containing thermal and visible folders
import os

def find_llvip_root(start_dir):
    for root, dirs, files in os.walk(start_dir):
        if "thermal" in dirs and "visible" in dirs:
            return root
    return start_dir

llvip_path = find_llvip_root("/kaggle/working/data")
print(f" LLVIP Dataset Root set to: {llvip_path}")

# Export environment variable for the training scripts
os.environ["LLVIP_ROOT"] = llvip_path


Cloning into 'Major-project'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 49 (delta 14), reused 38 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 26.63 KiB | 1.11 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/kaggle/working/Major-project/Major-project
 LLVIP Dataset Root set to: /kaggle/working/data


In [4]:
# Reset directory to prevent nested folders
%cd /kaggle/working/
!rm -rf Major-project

# Clone fresh code
!git clone https://github.com/Akhila-Bijja/Major-project.git
%cd Major-project


/kaggle/working
Cloning into 'Major-project'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 49 (delta 14), reused 38 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 26.63 KiB | 1.11 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/kaggle/working/Major-project


In [5]:
%cd /kaggle/working/Major-project
!git pull origin main


/kaggle/working/Major-project
From https://github.com/Akhila-Bijja/Major-project
 * branch            main       -> FETCH_HEAD
Already up to date.


In [6]:
%cd /kaggle/working/Major-project
!git pull origin main


/kaggle/working/Major-project
From https://github.com/Akhila-Bijja/Major-project
 * branch            main       -> FETCH_HEAD
Already up to date.


In [7]:
%cd /kaggle/working/Major-project
!git pull origin main


/kaggle/working/Major-project
From https://github.com/Akhila-Bijja/Major-project
 * branch            main       -> FETCH_HEAD
Already up to date.


In [8]:
!python train_stage1_structure.py


--- STAGE 1: Training Structure Extraction Network (Module 3A) ---
[Train] Loaded Thermal from: /kaggle/input/datasets/akhilabijja/llvip-images/LLVIP/infrared/train
[Train] Loaded Visible from: /kaggle/input/datasets/akhilabijja/llvip-images/LLVIP/visible/train
Epoch [1/15] - Structure Loss: 0.3691
Epoch [2/15] - Structure Loss: 0.3292
Epoch [3/15] - Structure Loss: 0.2805
Epoch [4/15] - Structure Loss: 0.2398
Epoch [5/15] - Structure Loss: 0.2191
Epoch [6/15] - Structure Loss: 0.2082
Epoch [7/15] - Structure Loss: 0.2020
Epoch [8/15] - Structure Loss: 0.1979
Epoch [9/15] - Structure Loss: 0.1946
Epoch [10/15] - Structure Loss: 0.1923
Epoch [11/15] - Structure Loss: 0.1905
Epoch [12/15] - Structure Loss: 0.1890
Epoch [13/15] - Structure Loss: 0.1875
Epoch [14/15] - Structure Loss: 0.1863
Epoch [15/15] - Structure Loss: 0.1851
Structure Model saved successfully at: ./checkpoints/structure_model_stage1.pth


In [9]:
# Step 1: Verify GPU & LLVIP Dataset Path
import os
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

# Check LLVIP Dataset presence on Kaggle
dataset_root = "/kaggle/input/llvip-dataset/LLVIP"
if not os.path.exists(dataset_root):
    print(f"⚠️ Warning: LLVIP dataset not found at {dataset_root}.")
    print("Please click '+ Add Data' in the right panel and attach the LLVIP dataset.")
else:
    print(f"✅ LLVIP Dataset detected at: {dataset_root}")
    print("Thermal train count:", len(os.listdir(os.path.join(dataset_root, 'thermal', 'train'))))
    print("Visible train count:", len(os.listdir(os.path.join(dataset_root, 'visible', 'train'))))

CUDA Available: True
Using GPU: Tesla T4
⚠️ Warning: LLVIP dataset not found at /kaggle/input/llvip-dataset/LLVIP.
Please click '+ Add Data' in the right panel and attach the LLVIP dataset.


--- 
## Stage 1: Pre-train Structure Extraction Network (Module 3A)
Extracts edge maps, contours, and high-frequency structural features from thermal inputs.

In [10]:
!python train_stage1_structure.py

--- STAGE 1: Training Structure Extraction Network (Module 3A) ---
[Train] Loaded Thermal from: /kaggle/input/datasets/akhilabijja/llvip-images/LLVIP/infrared/train
[Train] Loaded Visible from: /kaggle/input/datasets/akhilabijja/llvip-images/LLVIP/visible/train
Epoch [1/15] - Structure Loss: 0.3691
Epoch [2/15] - Structure Loss: 0.3320
Epoch [3/15] - Structure Loss: 0.2846
Epoch [4/15] - Structure Loss: 0.2426
Epoch [5/15] - Structure Loss: 0.2205
Epoch [6/15] - Structure Loss: 0.2096
Epoch [7/15] - Structure Loss: 0.2037
Epoch [8/15] - Structure Loss: 0.1988
Epoch [9/15] - Structure Loss: 0.1956
Epoch [10/15] - Structure Loss: 0.1931
Epoch [11/15] - Structure Loss: 0.1910
Epoch [12/15] - Structure Loss: 0.1895
Epoch [13/15] - Structure Loss: 0.1880
Epoch [14/15] - Structure Loss: 0.1868
Epoch [15/15] - Structure Loss: 0.1855
Structure Model saved successfully at: ./checkpoints/structure_model_stage1.pth


--- 
## Stage 2: Train Feature Fusion (4) & Appearance Generator (5)
Fuses structural (3A) and semantic (3B) features to synthesize realistic visible RGB images.

In [11]:
!python train_stage2_generator.py

--- STAGE 2: Training Fusion Module & Appearance Generator (Modules 4 & 5) ---
[Train] Loaded Thermal from: /kaggle/input/datasets/akhilabijja/llvip-images/LLVIP/infrared/train
[Train] Loaded Visible from: /kaggle/input/datasets/akhilabijja/llvip-images/LLVIP/visible/train
Loaded Structure model weights from ./checkpoints/structure_model_stage1.pth
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 167MB/s]
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|█████████████████████████████████████████| 528M/528M [00:03<00:00, 166MB/s]
Traceback (most recent call last):
  File "/kaggle/working/Major-project/train_stage2_generator.py", line 147, in <module>
    train_generator_stage()
    ~~~~~~~~~~~~~~~~~~~~~^^
  File "/kaggle/working/Major-project/train_stag

--- 
## Stage 3: End-to-End Joint Fine-Tuning
Unfreezes all modules (3A, 3B, 4, 5) and optimizes the full system jointly.

In [12]:
!python train_stage3_end2end.py

--- STAGE 3: End-to-End Joint Fine-Tuning ---
[Train] Loaded Thermal from: /kaggle/input/datasets/akhilabijja/llvip-images/LLVIP/infrared/train
[Train] Loaded Visible from: /kaggle/input/datasets/akhilabijja/llvip-images/LLVIP/visible/train
Traceback (most recent call last):
  File "/kaggle/working/Major-project/train_stage3_end2end.py", line 141, in <module>
    train_end2end()
    ~~~~~~~~~~~~~^^
  File "/kaggle/working/Major-project/train_stage3_end2end.py", line 94, in train_end2end
    fused_feats = fusion_module(thermal_imgs, struct_feats, sem_feats)
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1790, in _call_impl
    return forward_call(*args, **kwargs)
  File "/kaggle/working/Major-project/models/fusion_module.py", line 50, in forward
    cat_feat

--- 
## Stage 4: Quantitative & Qualitative Evaluation
Computes PSNR and SSIM metrics and displays side-by-side comparison images.

In [13]:
!python evaluate.py

# Display sample output grid inline in notebook
import glob
from PIL import Image
import matplotlib.pyplot as plt

results = sorted(glob.glob("./results/*.jpg")) + sorted(glob.glob("./results/*.png"))
if results:
    print(f"Total Generated Sample Grids: {len(results)}")
    fig, axes = plt.subplots(min(3, len(results)), 1, figsize=(15, 10))
    if len(results) == 1:
        axes = [axes]
    for i, img_path in enumerate(results[:3]):
        img = Image.open(img_path)
        axes[i].imshow(img)
        axes[i].set_title(f"Sample {i+1}: Thermal | Structure | Generated Visible | Ground Truth RGB | Confidence")
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

--- Evaluating Model on Test Dataset ---
[Test] Loaded Thermal from: /kaggle/input/datasets/akhilabijja/llvip-images/LLVIP/infrared/test
[Test] Loaded Visible from: /kaggle/input/datasets/akhilabijja/llvip-images/LLVIP/visible/test
Traceback (most recent call last):
  File "/kaggle/working/Major-project/evaluate.py", line 100, in <module>
    evaluate()
    ~~~~~~~~^^
  File "/kaggle/working/Major-project/evaluate.py", line 64, in evaluate
    fused = fusion_module(thermal, struct_feats, sem_feats)
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1790, in _call_impl
    return forward_call(*args, **kwargs)
  File "/kaggle/working/Major-project/models/fusion_module.py", line 50, in forward
    cat_features = torch.cat([thermal_input, structure_features, semant